# Task 2 routed validation comparison
Attach private datasets containing the extracted classifier, salt, blur, occlusion, and selected Task1 trial005 run folders. Use best.pt from each complete run folder, not older MLflow copies. This notebook evaluates validation only and does not train. Enable Internet and GPU; CPU is supported but slower. Run cells individually and set paths before evaluation.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
assert (repo / 'evaluation/evaluate_task2.py').is_file()
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
for path in Path('/kaggle/input').rglob('best.pt'):
    if '/mlruns/' not in str(path): print(path)


In [ ]:
# Replace all five paths with paths printed above.
checkpoints = {
    'classifier': Path('/kaggle/input/REPLACE/classifier/best.pt'),
    'salt': Path('/kaggle/input/REPLACE/salt/best.pt'),
    'blur': Path('/kaggle/input/REPLACE/blur/best.pt'),
    'occlusion': Path('/kaggle/input/REPLACE/occlusion/best.pt'),
    'task1': Path('/kaggle/input/REPLACE/trial_005/best.pt'),
}
for name,path in checkpoints.items():
    assert path.is_file(), f'Set the exact path for {name}: {path}'
    saved = torch.load(path, map_location='cpu', weights_only=True)
    print(name, 'epoch:', saved['progress']['epoch'], 'condition:', saved['config'].get('condition'))
    assert not saved['data_signature']['train_limit'] and not saved['data_signature']['val_images'], 'Do not use debug weights'


In [ ]:
output = Path('/kaggle/working/task2-routing-validation')
command = [sys.executable, '-m', 'evaluation.evaluate_task2', '--device', 'cuda',
           '--alpha', '0.8', '--output-dir', str(output)]
for name,path in checkpoints.items():
    command += [f'--{name}-checkpoint', str(path)]
subprocess.run(command, cwd=repo, check=True)


In [ ]:
import shutil
from IPython.display import display, Markdown, Image, FileLink
display(Markdown((output / 'README.md').read_text()))
display(Image(filename=str(output / 'representative_grid.png')))
backup = shutil.make_archive('/kaggle/working/task2_routing_backup', 'zip', output.parent, output.name)
display(FileLink(backup))


Download task2_routing_backup.zip before ending the session and upload a copy to Drive manually. Share summary.json, representative_grid.png and routing_failures.png (if generated). The same alpha0.8 applies to all reconstruction scores regardless of training alpha. Source/data mismatches fail explicitly; do not bypass checks. Use a new output directory if rerunning. Official test data remains untouched; tuning and final evaluation remain pending.
